### 参数管理

In [64]:
import torch
from torch import nn
from torch.nn import functional as F

In [39]:
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
X = torch.rand(2,4)
net(X)

tensor([[0.1848, 0.1289],
        [0.0404, 0.2804]], grad_fn=<AddmmBackward0>)

参数访问

In [40]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.3411,  0.3368, -0.1338, -0.3418,  0.2566,  0.0558,  0.3452, -0.0008],
        [ 0.1943, -0.2471,  0.2449,  0.1372, -0.1093,  0.2314, -0.2334, -0.3120]])), ('bias', tensor([0.2852, 0.1506]))])


目标参数

In [41]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)
print(net[2].weight.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.2852, 0.1506], requires_grad=True)
tensor([0.2852, 0.1506])
tensor([[-0.3411,  0.3368, -0.1338, -0.3418,  0.2566,  0.0558,  0.3452, -0.0008],
        [ 0.1943, -0.2471,  0.2449,  0.1372, -0.1093,  0.2314, -0.2334, -0.3120]])


In [42]:
net[2].weight.grad == None

True

一次性访问所有参数

In [ ]:
# 访问第一个连接层
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
# 访问所有层
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([2, 8])) ('2.bias', torch.Size([2]))


In [44]:
net.state_dict()['2.bias'].data

tensor([0.2852, 0.1506])

从嵌套块中收集参数

In [45]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}', block1())
    return net
rgnet = nn.Sequential(block2(), nn.Linear(4,1))
rgnet(X)

tensor([[-0.4938],
        [-0.4939]], grad_fn=<AddmmBackward0>)

In [46]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [47]:
print(f'第一个块中的第二个子块的第一个偏置：{rgnet[0][1][0].bias.data}')
print(f'第一个块中的第二个子块的第一个偏置：{rgnet[0][1][0].weight.data}')
print(f'第二个块中的权重:{rgnet[1].weight.data}')
print(f'第二个块中的偏置:{rgnet[1].bias.data}')

第一个块中的第二个子块的第一个偏置：tensor([-0.0997,  0.0693,  0.4464,  0.3611, -0.0806, -0.2656, -0.3685, -0.4596])
第一个块中的第二个子块的第一个偏置：tensor([[ 0.3743,  0.2321, -0.3812, -0.0836],
        [ 0.2376,  0.1854, -0.3021, -0.4010],
        [ 0.2320,  0.2329, -0.1690, -0.0795],
        [-0.3488, -0.0216,  0.0243,  0.2503],
        [-0.1941,  0.0549, -0.4368,  0.2317],
        [-0.1023,  0.4305, -0.3575,  0.3646],
        [-0.2668,  0.1697,  0.0248, -0.4415],
        [-0.1928,  0.1195, -0.2840, -0.3927]])
第二个块中的权重:tensor([[-0.4437, -0.3307, -0.0615, -0.2151]])
第二个块中的偏置:tensor([-0.2803])


#### 参数初始化

内置初始化

In [48]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 8.6460e-03,  2.7274e-03, -9.1820e-05,  4.2095e-03]), tensor(0.))

In [50]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [52]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)
net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data)
print(net[2].weight.data)

tensor([[ 0.6197,  0.1737,  0.6839, -0.2637],
        [-0.6517,  0.5348,  0.3363, -0.6360],
        [-0.5237,  0.2189, -0.6857,  0.0790],
        [-0.2172,  0.1830, -0.0432, -0.3294],
        [-0.2473,  0.2120, -0.5750, -0.5354],
        [-0.0102,  0.1035, -0.2198,  0.4500],
        [-0.2974, -0.5293, -0.5277,  0.3527],
        [-0.3757, -0.0575, -0.6505, -0.3000]])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.],
        [42., 42., 42., 42., 42., 42., 42., 42.]])


自定义初始化

In [57]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape) for name, param in m.named_parameters()])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 0.5

net.apply(my_init)
net[0].weight[:2]

Init ('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
Init ('weight', torch.Size([2, 8])) ('bias', torch.Size([2]))


tensor([[-5.9336,  3.1112, -0.5567, -9.5144],
        [ 6.8536, -1.7476, -7.3248,  7.5859]], grad_fn=<SliceBackward0>)

我倒要看看怎么今天就矫情了

In [55]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape) for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([2, 8])


tensor([[-7.3856, -0.0000, -0.0000, -9.9088],
        [-8.6139,  0.0000,  0.0000,  0.0000]], grad_fn=<SliceBackward0>)

#### 参数绑定

In [61]:
shared = nn.Linear(8,8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 2), nn.ReLU())
net(X)
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
print(net[2].weight.data[0, 0] == net[4].weight.data[0, 0])

tensor([True, True, True, True, True, True, True, True])
tensor(True)


In [66]:
class MLP(nn.Module):
    # 模型参数声明层，这里我们声明两个全连接层
    def __init__(self):
        # 调用父类Module的初始化函数来必要的初始化，这样在类实例化时也可以指定别的参数
        super().__init__()
        self.hidden = nn.Linear(4, 256)
        self.output = nn.Linear(256, 4)
    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))  # 先hidden，再relu，再out
net = MLP()
net(X)

tensor([[ 0.1313, -0.1307, -0.0266, -0.2639],
        [ 0.0358, -0.0358,  0.1625, -0.2142]], grad_fn=<AddmmBackward0>)

In [78]:
print(f'隐藏层的形状：{net.hidden.weight.data.shape}')
print(f'隐藏层第一行：{net.hidden.weight.data[0]}')
print(f'输出的形状：{net.output.weight.data.shape}')
print(f'输出层：{net.output.weight.data}')

隐藏层的形状：torch.Size([256, 4])
隐藏层第一行：tensor([ 0.0657,  0.2765, -0.2024,  0.4044])
输出的形状：torch.Size([4, 256])
输出层：tensor([[ 0.0275, -0.0529, -0.0286,  ...,  0.0617, -0.0605,  0.0570],
        [-0.0518,  0.0308,  0.0077,  ..., -0.0544, -0.0578, -0.0237],
        [ 0.0553,  0.0347, -0.0234,  ..., -0.0130, -0.0018,  0.0085],
        [-0.0220,  0.0295, -0.0565,  ...,  0.0095, -0.0339, -0.0423]])
